# Exercise 5.2 — A 3D cube with Model, View and Projection matrices

**Cube.** Only coordinates −1 and 1 are used. A cube has 6 faces; each square face is 2 triangles, so
6 × 2 × 3 = **36 vertices**. Each of the 8 corners gets a different colour, so the colours blend across each face.

**The three matrices** (all made with glm):
* **Model** `M = translate(10, 10, 10)` — puts the cube at (10, 10, 10) in the world.
* **View** `V = lookAt(cameraPosition, cameraTarget, up)` — our camera is at (15, 14, 17), looks at the cube
  (10, 10, 10), with `up = (0, 1, 0)` (camera held straight).
* **Projection** `P = perspective(45°, width/height, 0.1, 100)` — field of view, aspect ratio, near and far planes.

We multiply them on the CPU: $MVP = P \cdot V \cdot M$ and send the result to the shader as the uniform `MVP`
(`gl.uniformMatrix4fv`). The shader then only does `gl_Position = MVP * vec4(position, 1)` for each vertex.

We also rotate the cube slowly (`M = translate · rotate`) so you can see it is really 3D.

## Library
Only **glm-js** is needed (for matrices). The WebGL code is written directly, so this notebook still works
even when the course JS server is down.

In [ ]:
%%html
<script src='https://git.io/glm-js.min.js'></script>

In [ ]:
%%html
<script id="ex52-vs" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
varying vec3 ex_Color;
void main(void) {
    gl_Position = MVP * vec4(in_Position, 1.0);
    ex_Color = in_Color;
}
</script>

In [ ]:
%%html
<script id="ex52-fs" type="x-shader/x-fragment">
precision mediump float;
varying vec3 ex_Color;
void main(void) {
    gl_FragColor = vec4(ex_Color, 1.0);
}
</script>

In [ ]:
%%html
<script>
// ---------- small WebGL helpers ----------
// compile the two shaders (read from the <script> tags) and link them into a program
window.makeProgram = function (gl, vsId, fsId) {
  const compile = (type, id) => {
    const s = gl.createShader(type);
    gl.shaderSource(s, document.getElementById(id).text);
    gl.compileShader(s);
    if (!gl.getShaderParameter(s, gl.COMPILE_STATUS)) console.error(id, gl.getShaderInfoLog(s));
    return s;
  };
  const p = gl.createProgram();
  gl.attachShader(p, compile(gl.VERTEX_SHADER, vsId));
  gl.attachShader(p, compile(gl.FRAGMENT_SHADER, fsId));
  gl.linkProgram(p);
  return p;
};
// copy a JS array of numbers into a GPU buffer (GL wants 32-bit floats)
window.makeBuffer = function (gl, data) {
  const b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
};
// connect a buffer (3 floats per vertex) to an attribute of the shader
window.useAttrib = function (gl, prog, name, buffer) {
  const loc = gl.getAttribLocation(prog, name);
  if (loc < 0) return;
  gl.bindBuffer(gl.ARRAY_BUFFER, buffer);
  gl.vertexAttribPointer(loc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(loc);
};
// call render(time) on every animation frame
window.animate = function (render) {
  const step = (time) => { render(time); requestAnimationFrame(step); };
  requestAnimationFrame(step);
};
// ---------- cube: 6 faces x 2 triangles x 3 vertices = 36 vertices ----------
// only coordinates -1 and 1 are used; the 8 corners are repeated for every face
window.cubeData = function () {
  const p = [[-1,-1,-1], [ 1,-1,-1], [ 1, 1,-1], [-1, 1,-1],
             [-1,-1, 1], [ 1,-1, 1], [ 1, 1, 1], [-1, 1, 1]];
  // one different colour per corner (red, green, blue, yellow, ...)
  const c = [[1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,0.5,0], [0.5,0.5,0.5]];
  const faces = [[4,5,6,7], [1,0,3,2], [0,4,7,3], [5,1,2,6], [7,6,2,3], [0,1,5,4]];
  const pos = [], col = [];
  for (const [a, b, cc, d] of faces)
    for (const i of [a, b, cc, a, cc, d]) {      // square face = 2 triangles
      pos.push(...p[i]);
      col.push(...c[i]);
    }
  return { pos, col, n: pos.length / 3 };
};
</script>

In [ ]:
%%html
<canvas id="ex52Canvas" width="500" height="500" style="border:2px solid #000000;"></canvas>
<div id="ex52CanvasInfo" style="font-family:monospace"></div>
<script>
(function () {
  const cv = document.getElementById("ex52Canvas");
  const gl = cv.getContext("webgl");
  const prog = makeProgram(gl, "ex52-vs", "ex52-fs");
  const cube = cubeData();
  const posBuf = makeBuffer(gl, cube.pos), colBuf = makeBuffer(gl, cube.col);
  const uMVP = gl.getUniformLocation(prog, "MVP");
  gl.enable(gl.DEPTH_TEST);
  gl.clearColor(1, 1, 1, 1);

  // camera (view) and projection do not change, so we compute them once
  const V = glm.lookAt(glm.vec3(15, 14, 17), glm.vec3(10, 10, 10), glm.vec3(0, 1, 0));
  const P = glm.perspective(glm.radians(45), cv.width / cv.height, 0.1, 100);

  animate((time) => {
    gl.clear(gl.COLOR_BUFFER_BIT | gl.DEPTH_BUFFER_BIT);
    // model: first rotate around the cube centre, then move it to (10,10,10)
    const T = glm.translate(glm.mat4(1), glm.vec3(10, 10, 10));
    const R = glm.rotate(glm.mat4(1), time / 1000, glm.vec3(0, 1, 0));
    const M = T['*'](R);
    const MVP = P['*'](V['*'](M));

    gl.useProgram(prog);
    useAttrib(gl, prog, "in_Position", posBuf);
    useAttrib(gl, prog, "in_Color", colBuf);
    gl.uniformMatrix4fv(uMVP, false, MVP.elements);   // send the matrix to the shader
    gl.drawArrays(gl.TRIANGLES, 0, cube.n);
    document.getElementById("ex52CanvasInfo").textContent = "cube at (10,10,10), " + cube.n + " vertices";
  });
})();
</script>